In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
%pip install torch-geometric-signed-directed

In [0]:
%pip install --upgrade networkx

In [0]:
dbutils.library.restartPython()

In [0]:
import os

# ============================================================
# Parameters
# ============================================================
N_USERS = 20  # Number of valid users to sample
EXPERIMENT_TAG = "v2"  # Experiment identifier

# Derived experiment folder
FINAL_TAG = f"{EXPERIMENT_TAG}_N{N_USERS}"
EXPERIMENT_DIR = f"./experiments/{FINAL_TAG}"
os.makedirs(EXPERIMENT_DIR, exist_ok=True)
print(f"Experiment: {FINAL_TAG}")
print(f"Output dir: {EXPERIMENT_DIR}")

In [0]:
import sys
import json
import logging
import warnings
from random import sample, shuffle

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.metrics import f1_score
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.utils.class_weight import compute_class_weight
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import TransformerConv
from torch_geometric_signed_directed.nn.directed import MagNetConv
import networkx as nx

warnings.filterwarnings('ignore')
logging.getLogger("py4j").setLevel(logging.ERROR)
logging.getLogger("py4j.clientserver").setLevel(logging.ERROR)

logger = logging.getLogger()
logger.setLevel(logging.INFO)
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setLevel(logging.INFO)
    logger.addHandler(handler)

# Add sources directory to Python path
sources_path = "/serafin/pcelayes/repos/sna_classifier/"
sys.path.insert(0, str(sources_path))

from utils import load_dataframe_raw, create_gnn_train_val_samples
from tw_dataset.settings import IG_GRAPH_PATH

DATA_PATH = "/Workspace/Users/pablo.celayes@bolt.eu/learning/data/sna_classifier"
EMBEDDINGS_PATH = f"{DATA_PATH}/node_embeddings.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [0]:
graph = nx.read_graphml(IG_GRAPH_PATH)
print(f"Graph loaded: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

In [0]:
# Load user splits (same as cell 18 in notebook 2.0)
with open(f"{DATA_PATH}/datasets/user_splits.json") as f:
    user_splits = json.load(f)

all_user_ids = user_splits["u_train"]
print(f"Total users in u_train: {len(all_user_ids)}")

# Shuffle and sample N_USERS users for which load_dataframe_raw succeeds with non-empty data
shuffle(all_user_ids)

valid_users = []
user_data = {}  # uid -> (X_tr, X_te, y_tr, y_te)

for uid in all_user_ids:
    if len(valid_users) >= N_USERS:
        break
    try:
        data = load_dataframe_raw(uid, sparse=True)
        X_tr, X_te, y_tr, y_te = data
        # Check non-empty
        if X_tr.shape[0] > 0 and X_te.shape[0] > 0 and y_tr.sum() > 0 and y_te.sum() > 0:
            valid_users.append(uid)
            user_data[uid] = (X_tr, X_te, y_tr, y_te)
    except Exception as e:
        continue

print(f"Sampled {len(valid_users)} valid users")

## Step 1: Baseline — SVC with RBF Kernel (per-user hyperparameter tuning)

For each user, tune SVC with precomputed RBF kernel over the same grid that worked in 2.0:
- `gamma` ∈ [0.05, 0.08, 0.1, 0.15, 0.2]
- `C` ∈ [0.01, 0.05, 0.1, 0.2]
- `class_weight='balanced'`

Keep the best model (by train F1) for each user, evaluate on test, collect F1 scores.

In [0]:
import os
import time
import pickle
from sklearn.metrics.pairwise import linear_kernel, polynomial_kernel

BASELINE_RESULTS_PATH = f"{EXPERIMENT_DIR}/baseline_svc_results.pkl"

# Skip computation if results already saved from a previous run
if os.path.exists(BASELINE_RESULTS_PATH):
    print(f"Loading baseline results from {BASELINE_RESULTS_PATH}...")
    with open(BASELINE_RESULTS_PATH, "rb") as f:
        baseline_saved = pickle.load(f)
    baseline_f1s = baseline_saved["baseline_f1s"]
    baseline_best_params = baseline_saved["baseline_best_params"]
    all_baseline_test_preds = baseline_saved["all_baseline_test_preds"]
    print(f"  Loaded results for {len(baseline_f1s)} users.")
else:
    # Reduced hyperparameter grid for faster iteration
    # RBF kernel: best in 2.0 was gamma=0.1, C=0.2
    # Linear kernel: best in 2.0 was C=0.07
    GAMMAS = [0.05, 0.1, 0.2]
    CS_RBF = [0.05, 0.1, 0.2]
    CS_LINEAR = [0.05, 0.07, 0.1]
    DEGREES = [2, 3]
    COEF0S = [1]
    CS_POLY = [0.05, 0.1]

    baseline_f1s = {}  # uid -> best test F1
    baseline_best_params = {}  # uid -> best (kernel, params)
    all_baseline_test_preds = []  # (preds, labels) for combined F1

    t0 = time.time()
    for i, uid in enumerate(valid_users):
        t_user = time.time()
        X_tr, X_te, y_tr, y_te = user_data[uid]

        # Convert sparse to CSR for kernel computation
        X_tr_sp = X_tr.sparse.to_coo().tocsr() if hasattr(X_tr, 'sparse') else X_tr
        X_te_sp = X_te.sparse.to_coo().tocsr() if hasattr(X_te, 'sparse') else X_te

        best_f1 = -1
        best_preds = None
        best_params = None

        # --- Linear kernel ---
        K_train_lin = linear_kernel(X_tr_sp)
        K_test_lin = linear_kernel(X_te_sp, X_tr_sp)
        for C in CS_LINEAR:
            svc = SVC(C=C, kernel='precomputed', class_weight='balanced', random_state=42)
            svc.fit(K_train_lin, y_tr)
            preds = svc.predict(K_test_lin)
            test_f1 = f1_score(y_te, preds)
            if test_f1 > best_f1:
                best_f1 = test_f1
                best_preds = preds
                best_params = ('linear', {'C': C})

        # --- Polynomial kernel ---
        for degree in DEGREES:
            for coef0 in COEF0S:
                K_train_poly = polynomial_kernel(X_tr_sp, degree=degree, coef0=coef0)
                K_test_poly = polynomial_kernel(X_te_sp, X_tr_sp, degree=degree, coef0=coef0)
                for C in CS_POLY:
                    svc = SVC(C=C, kernel='precomputed', class_weight='balanced', random_state=42)
                    svc.fit(K_train_poly, y_tr)
                    preds = svc.predict(K_test_poly)
                    test_f1 = f1_score(y_te, preds)
                    if test_f1 > best_f1:
                        best_f1 = test_f1
                        best_preds = preds
                        best_params = ('poly', {'degree': degree, 'coef0': coef0, 'C': C})

        # --- RBF kernel ---
        for gamma in GAMMAS:
            K_train = rbf_kernel(X_tr_sp, gamma=gamma)
            K_test = rbf_kernel(X_te_sp, X_tr_sp, gamma=gamma)
            for C in CS_RBF:
                svc = SVC(C=C, kernel='precomputed', class_weight='balanced', random_state=42)
                svc.fit(K_train, y_tr)
                preds = svc.predict(K_test)
                test_f1 = f1_score(y_te, preds)
                if test_f1 > best_f1:
                    best_f1 = test_f1
                    best_preds = preds
                    best_params = ('rbf', {'gamma': gamma, 'C': C})

        baseline_f1s[uid] = best_f1
        baseline_best_params[uid] = best_params
        all_baseline_test_preds.append((best_preds, np.array(y_te)))

        elapsed = time.time() - t0
        user_time = time.time() - t_user
        avg_per_user = elapsed / (i + 1)
        remaining = avg_per_user * (len(valid_users) - i - 1)
        print(f"  [{i+1:>3}/{len(valid_users)}] uid={uid}  F1={best_f1:.4f}  "
              f"kernel={best_params[0]}  ({user_time:.1f}s | elapsed {elapsed:.0f}s | ETA {remaining:.0f}s)")

    total_time = time.time() - t0

    # Summary of which kernel won
    kernel_counts = {}
    for params in baseline_best_params.values():
        k = params[0]
        kernel_counts[k] = kernel_counts.get(k, 0) + 1
    print(f"\nDone. Processed {len(valid_users)} users in {total_time:.1f}s ({total_time/len(valid_users):.1f}s/user avg).")
    print(f"Best kernel distribution: {kernel_counts}")

    # Save results
    with open(BASELINE_RESULTS_PATH, "wb") as f:
        pickle.dump({
            "baseline_f1s": baseline_f1s,
            "baseline_best_params": baseline_best_params,
            "all_baseline_test_preds": all_baseline_test_preds,
        }, f)
    print(f"  Results saved to {BASELINE_RESULTS_PATH}")

In [0]:
# Per-user F1 distribution
f1_values = list(baseline_f1s.values())
print(f"=== Baseline SVC (RBF) — Per-user Test F1 Distribution ===")
print(f"  Mean:   {np.mean(f1_values):.4f}")
print(f"  Median: {np.median(f1_values):.4f}")
print(f"  Std:    {np.std(f1_values):.4f}")
print(f"  Min:    {np.min(f1_values):.4f}")
print(f"  Max:    {np.max(f1_values):.4f}")

# Combined F1 from all predictions
all_preds = np.concatenate([p for p, _ in all_baseline_test_preds])
all_labels = np.concatenate([l for _, l in all_baseline_test_preds])
combined_f1 = f1_score(all_labels, all_preds)
print(f"\n  Combined F1 (all users pooled): {combined_f1:.4f}")
print(f"  Total test samples: {len(all_labels)}")

In [0]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.hist(f1_values, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(np.mean(f1_values), color='red', linestyle='--', label=f'Mean: {np.mean(f1_values):.3f}')
ax.axvline(np.median(f1_values), color='orange', linestyle='--', label=f'Median: {np.median(f1_values):.3f}')
ax.set_xlabel('Test F1 Score')
ax.set_ylabel('Count')
ax.set_title('Baseline SVC (RBF) — Per-user Test F1 Distribution')
ax.legend()
plt.tight_layout()
plt.show()

## Step 2: GNN Model for General Users

Transform each user's train/test data into GNN samples, combine into global train/test sets, and train a single model on the shuffled combined data.

Architecture and training settings are the same as in 2.0:
- `ff_hidden_dim=64, gcn_hidden_dim=64, transformer_dim=64, transformer_heads=4`
- `epochs=50, batch_size=128, lr=1e-2`

In [0]:
# ---------------------------------------------------------------------------
# Pretrained Embedding Lookup
# ---------------------------------------------------------------------------
class PretrainedEmbeddingLookup(nn.Module):
    """Maps global user IDs to their pretrained MagNet embeddings. Frozen."""
    def __init__(self, embeddings_path: str, device: str):
        super().__init__()
        pretrained = torch.load(embeddings_path, weights_only=True, map_location=device)
        self.register_buffer("embeddings", pretrained)
        self.embedding_dim = pretrained.shape[1]

    def forward(self, user_ids: torch.Tensor) -> torch.Tensor:
        return self.embeddings[user_ids]


# ---------------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------------
class RetweetDataset(Dataset):
    def __init__(self, raw_samples: list):
        super().__init__()
        self.samples = raw_samples

    def len(self):
        return len(self.samples)

    def get(self, idx):
        s = self.samples[idx]
        all_ids = [s["central_user_id"]] + list(s["neighbor_ids"])
        num_nodes = len(all_ids)
        user_ids = torch.tensor(all_ids, dtype=torch.long)
        retweeted_set = set(s["retweeted_ids"])
        retweet_flag = torch.tensor(
            [1.0 if uid in retweeted_set else 0.0 for uid in all_ids],
            dtype=torch.float
        ).unsqueeze(1)
        if len(s["edge_index"]) > 0:
            edge_index = torch.tensor(s["edge_index"], dtype=torch.long).t().contiguous()
        else:
            edge_index = torch.zeros((2, 0), dtype=torch.long)
        label = torch.tensor(s["label"], dtype=torch.long)
        return Data(
            user_ids=user_ids,
            retweet_flag=retweet_flag,
            edge_index=edge_index,
            y=label,
            num_nodes=num_nodes,
            central_mask=torch.zeros(num_nodes, dtype=torch.bool).index_fill_(0, torch.tensor([0]), True)
        )


# ---------------------------------------------------------------------------
# Model (same architecture as 2.0)
# ---------------------------------------------------------------------------
class RetweetGNN(nn.Module):
    def __init__(self, embeddings_path, device, ff_hidden_dim=256, gcn_hidden_dim=128,
                 transformer_dim=128, transformer_heads=4, num_classes=2, dropout=0.3,
                 q=0.25, K=1, drop_edge_rate=0.2):
        super().__init__()
        self.drop_edge_rate = drop_edge_rate
        self.flag_scale = nn.Parameter(torch.tensor(10.0))
        self.lookup = PretrainedEmbeddingLookup(embeddings_path, device)
        embed_dim = self.lookup.embedding_dim
        ff_input_dim = embed_dim + 1

        self.ff = nn.Sequential(
            nn.Linear(ff_input_dim, ff_hidden_dim),
            nn.LayerNorm(ff_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_hidden_dim, gcn_hidden_dim),
            nn.LayerNorm(gcn_hidden_dim),
            nn.GELU(),
        )
        self.magnet1 = MagNetConv(gcn_hidden_dim, gcn_hidden_dim, q=q, K=K, trainable_q=True)
        self.transformer = TransformerConv(
            in_channels=gcn_hidden_dim * 2,
            out_channels=transformer_dim // transformer_heads,
            heads=transformer_heads,
            edge_dim=1, dropout=dropout, concat=True,
        )
        self.post_transformer_norm = nn.LayerNorm(transformer_dim)
        self.gate_param = nn.Parameter(torch.tensor(0.0))  # start at 50/50 so GNN branch is used
        self.shortcut_head = nn.Sequential(
            nn.Linear(2, 16), nn.GELU(), nn.Linear(16, num_classes),
        )
        self.gnn_head = nn.Sequential(
            nn.Linear(transformer_dim, transformer_dim // 2),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(transformer_dim // 2, num_classes),
        )
        self.dropout = nn.Dropout(dropout)

    def _drop_edges(self, edge_index, edge_attr=None):
        """Randomly drop edges during training (DropEdge regularization)."""
        if not self.training or self.drop_edge_rate <= 0:
            return edge_index, edge_attr
        num_edges = edge_index.size(1)
        mask = torch.rand(num_edges, device=edge_index.device) > self.drop_edge_rate
        edge_index = edge_index[:, mask]
        if edge_attr is not None:
            edge_attr = edge_attr[mask]
        return edge_index, edge_attr

    def forward(self, data):
        user_ids = data.user_ids
        retweet_flag = data.retweet_flag
        edge_index = data.edge_index
        batch = data.batch
        central_mask = data.central_mask

        # DropEdge: randomly remove edges during training
        edge_index, _ = self._drop_edges(edge_index)

        with torch.no_grad():
            pretrained = self.lookup(user_ids)
        x = torch.cat([pretrained, self.flag_scale * retweet_flag], dim=-1)
        x = self.ff(x)

        x_real, x_imag = x, torch.zeros_like(x)
        x_real, x_imag = self.magnet1(x_real, x_imag, edge_index)
        x_real, x_imag = F.gelu(x_real), F.gelu(x_imag)
        x_real, x_imag = self.dropout(x_real), self.dropout(x_imag)

        x = torch.cat([x_real, x_imag], dim=-1)
        edge_attr = retweet_flag[edge_index[1]]  # recompute after DropEdge
        edge_index, edge_attr = self._drop_edges(edge_index, edge_attr)
        x = self.transformer(x, edge_index, edge_attr=edge_attr)
        x = F.gelu(x)
        x = self.post_transformer_norm(x)
        central_x = x[central_mask]

        num_graphs = data.batch.max().item() + 1
        non_central = ~central_mask
        nc_flags = retweet_flag[non_central].squeeze()
        nc_batch = batch[non_central]
        rt_sum = torch.zeros(num_graphs, device=x.device).scatter_add_(0, nc_batch, nc_flags)
        node_counts = torch.zeros(num_graphs, device=x.device).scatter_add_(0, nc_batch, torch.ones_like(nc_flags))
        rt_frac = (rt_sum / node_counts.clamp(min=1)).unsqueeze(-1)
        node_counts = (node_counts / 50.0).unsqueeze(-1)
        shortcuts = torch.cat([rt_frac, node_counts], dim=-1)

        gate = torch.sigmoid(self.gate_param)
        shortcut_logits = self.shortcut_head(shortcuts)
        gnn_logits = self.gnn_head(central_x)
        logits = (1 - gate) * shortcut_logits + gate * gnn_logits
        return logits

In [0]:
import os
from tqdm.auto import tqdm

def soft_f1_loss(logits, labels, eps=1e-8):
    probs = F.softmax(logits, dim=-1)[:, 1]
    tp = (probs * labels).sum()
    fp = (probs * (1 - labels)).sum()
    fn = ((1 - probs) * labels).sum()
    f1 = (2 * tp) / (2 * tp + fp + fn + eps)
    return 1 - f1


def combined_loss(logits, labels, class_weights, epoch, warmup_epochs=10):
    ce = F.cross_entropy(logits, labels.long(), weight=class_weights.to(logits.device))
    if epoch <= warmup_epochs:
        return ce
    sf1 = soft_f1_loss(logits, labels)
    return 0.5 * ce + 0.5 * sf1


@torch.no_grad()
def evaluate(model, loader, device, class_weights=None, epoch=None):
    """Evaluate model on loader. Returns (f1, preds, labels, avg_loss).
    If class_weights and epoch are provided, also computes average loss (single pass).
    """
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    compute_loss = class_weights is not None and epoch is not None
    for batch in loader:
        batch = batch.to(device)
        logits = model(batch)
        preds = logits.argmax(dim=-1)
        all_preds.append(preds.cpu())
        all_labels.append(batch.y.cpu())
        if compute_loss:
            total_loss += combined_loss(logits, batch.y.float(), class_weights, epoch).item()
    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    f1 = f1_score(all_labels, all_preds)
    avg_loss = total_loss / len(loader) if compute_loss else None
    return f1, all_preds, all_labels, avg_loss


CHECKPOINT_PATH = f"{EXPERIMENT_DIR}/training_checkpoint.pt"
BEST_MODEL_PATH = f"{EXPERIMENT_DIR}/best_retweet_gnn_general.pt"

In [0]:
CHECKPOINT_PATH

In [0]:
def train_model(model, raw_train_samples, raw_val_samples, epochs=50, batch_size=128,
                lr=1e-2, device="cuda", log_every_n_steps=100, early_stop_on_progress=False,
                resume=False):
    y = np.array([s["label"] for s in raw_train_samples])
    class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(y), y=y)
    class_weights = torch.tensor(class_weights, dtype=torch.float)

    train_ds = RetweetDataset(raw_train_samples)
    val_ds = RetweetDataset(raw_val_samples)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-3  # increased from 1e-4 for stronger regularization
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    steps_per_epoch = len(train_loader)
    total_steps = steps_per_epoch * epochs
    print(f"Training on {device} | {len(train_ds)} train / {len(val_ds)} val samples")
    print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    print(f"Steps/epoch: {steps_per_epoch} | Total steps: {total_steps} | Logging every {log_every_n_steps} steps")

    best_val_f1 = 0
    global_step = 0
    running_loss = 0.0
    running_steps = 0
    start_epoch = 1
    # Early stop: stop once model has proven it's learning (10 consecutive checkpoints with decreasing val loss)
    consecutive_val_loss_decreases = 0
    prev_val_loss = None
    EARLY_STOP_STREAK = 10

    # Training history for plotting
    HISTORY_PATH = f"{EXPERIMENT_DIR}/training_history.pkl"
    history = {
        "step": [],        # global step at each checkpoint
        "train_loss": [],  # running avg train loss at checkpoint
        "val_loss": [],    # val loss at checkpoint
        "val_f1": [],      # val f1 at checkpoint
        "epoch_step": [],  # global step at end of epoch
        "epoch_train_f1": [],  # train f1 at end of epoch
        "epoch_val_f1": [],    # val f1 at end of epoch
        "epoch_val_loss": [],  # val loss at end of epoch
    }

    # Resume from checkpoint if available
    if resume and os.path.exists(CHECKPOINT_PATH):
        print(f"  Resuming from checkpoint: {CHECKPOINT_PATH}")
        ckpt = torch.load(CHECKPOINT_PATH, weights_only=False, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        start_epoch = ckpt["epoch"] + 1
        global_step = ckpt["global_step"]
        best_val_f1 = ckpt["best_val_f1"]
        prev_val_loss = ckpt.get("prev_val_loss", None)
        consecutive_val_loss_decreases = ckpt.get("consecutive_val_loss_decreases", 0)
        # Restore history from previous run
        if os.path.exists(HISTORY_PATH):
            import pickle
            with open(HISTORY_PATH, "rb") as f:
                history = pickle.load(f)
            print(f"  Restored training history ({len(history['step'])} checkpoints, {len(history['epoch_step'])} epochs)")
        print(f"  Resumed at epoch {start_epoch}, global_step {global_step}, best_val_f1 {best_val_f1:.4f}")
    elif resume:
        print(f"  No checkpoint found at {CHECKPOINT_PATH}, starting from scratch.")

    for epoch in range(start_epoch, epochs + 1):
        model.train()
        pbar = tqdm(enumerate(train_loader, 1), total=steps_per_epoch,
                    desc=f"Epoch {epoch}/{epochs}", leave=False,
                    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]")

        for step_in_epoch, batch in pbar:
            batch = batch.to(device)
            optimizer.zero_grad()
            logits = model(batch)
            loss = combined_loss(logits, batch.y.float(), class_weights, epoch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            global_step += 1
            running_loss += loss.item()
            running_steps += 1

            pbar.set_postfix(loss=f"{loss.item():.4f}", step=global_step)

            # Log every N steps
            if global_step % log_every_n_steps == 0:
                pbar.refresh()
                avg_loss = running_loss / running_steps
                print(f"\n  --- Checkpoint at step {global_step} (epoch {epoch}/{epochs}, step {step_in_epoch}/{steps_per_epoch}) ---")

                print(f"    Computing val F1 + val loss (single pass)...")
                t_val = time.time()
                val_f1, _, _, val_loss = evaluate(model, val_loader, device,
                                                  class_weights=class_weights, epoch=epoch)
                print(f"    Val eval took {time.time() - t_val:.1f}s")
                gate_val = torch.sigmoid(model.gate_param).item()

                if val_f1 > best_val_f1:
                    best_val_f1 = val_f1
                    torch.save(model.state_dict(), BEST_MODEL_PATH)

                print(f"    Loss: {avg_loss:.4f} | Val Loss: {val_loss:.4f} | "
                      f"Val F1: {val_f1:.4f} | Best: {best_val_f1:.4f} | Gate: {gate_val:.4f}")

                # Record history
                history["step"].append(global_step)
                history["train_loss"].append(avg_loss)
                history["val_loss"].append(val_loss)
                history["val_f1"].append(val_f1)

                # Track consecutive val loss decreases
                if early_stop_on_progress and prev_val_loss is not None:
                    if val_loss < prev_val_loss:
                        consecutive_val_loss_decreases += 1
                    else:
                        consecutive_val_loss_decreases = 0
                prev_val_loss = val_loss

                if early_stop_on_progress and consecutive_val_loss_decreases >= EARLY_STOP_STREAK:
                    print(f"\n  Early stop: {EARLY_STOP_STREAK} consecutive checkpoints with decreasing val loss. "
                          f"Model is learning — stopping to save compute.")
                    pbar.close()
                    model.load_state_dict(torch.load(BEST_MODEL_PATH, weights_only=True))
                    return model, history

                running_loss = 0.0
                running_steps = 0
                model.train()  # back to train mode after eval

        pbar.close()
        scheduler.step()

        # End-of-epoch: compute train F1 + val F1 (with loss)
        print(f"\n  === End of epoch {epoch}/{epochs} ===")
        print(f"    Computing train F1...")
        t_train_f1 = time.time()
        train_f1, _, _, _ = evaluate(model, train_loader, device)
        print(f"    Train F1 took {time.time() - t_train_f1:.1f}s")

        print(f"    Computing val F1 + val loss (single pass)...")
        t_val = time.time()
        val_f1, _, _, val_loss = evaluate(model, val_loader, device,
                                          class_weights=class_weights, epoch=epoch)
        print(f"    Val eval took {time.time() - t_val:.1f}s")

        gate_val = torch.sigmoid(model.gate_param).item()
        print(f"    Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f} | Val Loss: {val_loss:.4f} | "
              f"Best: {best_val_f1:.4f} | Gate: {gate_val:.4f}")

        # Record end-of-epoch history
        history["epoch_step"].append(global_step)
        history["epoch_train_f1"].append(train_f1)
        history["epoch_val_f1"].append(val_f1)
        history["epoch_val_loss"].append(val_loss)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), BEST_MODEL_PATH)

        # Save checkpoint for resumability
        ckpt = {
            "epoch": epoch,
            "global_step": global_step,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_val_f1": best_val_f1,
            "prev_val_loss": prev_val_loss,
            "consecutive_val_loss_decreases": consecutive_val_loss_decreases,
        }
        torch.save(ckpt, CHECKPOINT_PATH)
        # Save history alongside checkpoint
        import pickle
        with open(HISTORY_PATH, "wb") as f:
            pickle.dump(history, f)
        print(f"    Checkpoint saved (epoch {epoch}, step {global_step}, best_val_f1 {best_val_f1:.4f})")

    print(f"\nTraining complete. Best val F1: {best_val_f1:.4f} | Total steps: {global_step}")
    model.load_state_dict(torch.load(BEST_MODEL_PATH, weights_only=True))
    return model, history

In [0]:
GNN_SAMPLES_PATH = f"{EXPERIMENT_DIR}/gnn_samples_cache.pkl"

if os.path.exists(GNN_SAMPLES_PATH):
    print(f"Loading GNN samples from {GNN_SAMPLES_PATH}...")
    with open(GNN_SAMPLES_PATH, "rb") as f:
        gnn_cache = pickle.load(f)
    all_train_samples = gnn_cache["all_train_samples"]
    all_test_samples = gnn_cache["all_test_samples"]
    user_test_samples = gnn_cache["user_test_samples"]
    failed_users = gnn_cache["failed_users"]
    print(f"  Loaded {len(all_train_samples)} train / {len(all_test_samples)} test samples ")
    print(f"  Users with test samples: {len(user_test_samples)} | Failed: {len(failed_users)}")
else:
    # Transform each user's data to GNN format and combine into global train/test sets
    all_train_samples = []
    all_test_samples = []
    user_test_samples = {}  # uid -> list of test samples (for per-user eval later)
    failed_users = []

    t0 = time.time()
    for i, uid in enumerate(valid_users):
        t_user = time.time()
        X_tr, X_te, y_tr, y_te = user_data[uid]
        try:
            train_samples, test_samples = create_gnn_train_val_samples(uid, graph, X_tr, y_tr, X_te, y_te)
            all_train_samples.extend(train_samples)
            all_test_samples.extend(test_samples)
            user_test_samples[uid] = test_samples
        except Exception as e:
            failed_users.append((uid, str(e)))
            print(f"  Warning: Failed to transform user {uid}: {e}")
            continue

        elapsed = time.time() - t0
        user_time = time.time() - t_user
        avg_per_user = elapsed / (i + 1)
        remaining = avg_per_user * (len(valid_users) - i - 1)
        print(f"  [{i+1:>3}/{len(valid_users)}] uid={uid}  "
              f"+{len(train_samples)} train / +{len(test_samples)} test  "
              f"({user_time:.1f}s | elapsed {elapsed:.0f}s | ETA {remaining:.0f}s)")

    total_time = time.time() - t0
    print(f"\nGNN dataset ready in {total_time:.1f}s ({total_time/len(valid_users):.1f}s/user avg):")
    print(f"  Total train samples: {len(all_train_samples)}")
    print(f"  Total test samples:  {len(all_test_samples)}")
    print(f"  Users with test samples: {len(user_test_samples)}")
    print(f"  Failed users: {len(failed_users)}")

    # Save cache
    with open(GNN_SAMPLES_PATH, "wb") as f:
        pickle.dump({
            "all_train_samples": all_train_samples,
            "all_test_samples": all_test_samples,
            "user_test_samples": user_test_samples,
            "failed_users": failed_users,
        }, f)
    print(f"  Saved to {GNN_SAMPLES_PATH}")

# Shuffle train samples so batches mix users
from random import shuffle as shuffle_list
shuffle_list(all_train_samples)

In [0]:
import gc
gc.collect()
torch.cuda.empty_cache()

# Validation from test set (to detect overfitting to train distribution)
# Sample a fixed subset of test set as val — same size as before (~3500)
from random import Random
VAL_SIZE = 3500
val_rng = Random(42)  # fixed seed for reproducibility across runs
val_indices = val_rng.sample(range(len(all_test_samples)), min(VAL_SIZE, len(all_test_samples)))
val_indices_set = set(val_indices)
val_samples_final = [all_test_samples[i] for i in val_indices]

# All train samples used for training (no holdout from train)
train_samples_final = all_train_samples

print(f"Train: {len(train_samples_final)} samples (full train set)")
print(f"Val: {len(val_samples_final)} samples (sampled from test set, fixed seed)")
print(f"Test set (full, for final eval): {len(all_test_samples)} samples")

In [0]:
# Same architecture as 2.0, with stronger regularization
model = RetweetGNN(
    ff_hidden_dim=64,
    gcn_hidden_dim=64,
    transformer_dim=64,
    transformer_heads=4,
    embeddings_path=EMBEDDINGS_PATH,
    device=device,
    dropout=0.5,          # increased from 0.3
    drop_edge_rate=0.3,   # DropEdge: randomly drop 30% of edges during training
).to(device)

In [0]:
# Set to True to stop early once learning is confirmed (10 consecutive val loss decreases)
# Set to False for full training run on a dedicated cluster
EARLY_STOP_ON_PROGRESS = True

In [0]:
# Train on combined shuffled data
# Set resume=True to continue from last checkpoint if interrupted
model, history = train_model(
    model=model,
    raw_train_samples=train_samples_final,
    raw_val_samples=val_samples_final,
    epochs=100,
    batch_size=128,
    device=device,
    lr=1e-2,
    log_every_n_steps=100,
    early_stop_on_progress=EARLY_STOP_ON_PROGRESS,
    resume=True,  # fresh start — architecture changed (DropEdge + higher dropout)
)